# セッション間メモリを持つ旅行アシスタント

パーソナライズされたAI旅行アシスタントによるセッション間メモリの実世界デモンストレーション。このノートブックでは、Amazon S3 Vectorsを使用したマルチモーダル分析（テキスト、画像、PDF）と段階的なパーソナライゼーションを特徴とします。

## 学習内容

- 永続的なメモリを持つパーソナライズされた旅行アシスタントを構築する
- マルチモーダルコンテンツ（テキスト設定、画像、PDF）を処理する
- セッション間で段階的なパーソナライゼーションを実装する
- 本番環境対応のセッション間メモリを実証する

## 前提条件

- [ノートブック05: S3ベクトルメモリ](05-s3-vector-memory.ipynb)を完了していること
- [ノートブック07: 旅行コンテンツジェネレーター](07-travel-content-generator.ipynb)を使用してデモコンテンツを生成していること
- S3 Vectorバケットとインデックスを設定していること


## セットアップ

### 前提条件

1. `python travel_content_generator.py`を実行してデモアセットを作成する
2. S3 Vectorバケットを設定する（ノートブック04から）
3. AWS認証情報が設定されていることを確認する

In [ ]:
!pip install strands-agents strands-agents-tools boto3 -q

In [ ]:
import boto3
import os
from strands import Agent
from strands.models import BedrockModel
from strands_tools import image_reader, file_read
from video_reader_local import video_reader_local
from s3_memory import s3_vector_memory

print("✅ All imports successful!")

## 設定

In [ ]:
# AWS設定
AWS_REGION = 'us-east-1'
os.environ['AWS_REGION'] = AWS_REGION

# S3 Vectorsを設定（バケットとインデックスは自動的に作成されます）
os.environ['VECTOR_BUCKET_NAME'] = 'multimodal-vector-store'  # ⚠️ これを変更してください！
os.environ['VECTOR_INDEX_NAME'] = 'strands-multimodal'        # ⚠️ これを変更してください！
os.environ['EMBEDDING_MODEL'] = 'amazon.nova-2-multimodal-embeddings-v1:0'  # Nova埋め込み


## 旅行アシスタントエージェントの作成

次の機能を持つエージェントを作成します：
- **ベクトルメモリ**: セッション間の永続化
- **画像分析**: 目的地の写真
- **ドキュメント処理**: 旅程
- **パーソナライゼーション**: 保存された好みに基づく

In [ ]:
USER_ID = "demo_user_eli"  # メモリ分離のためのユーザーID

print(f"🌍 リージョン: {AWS_REGION}")
print(f"📦 ベクトルバケット: {os.environ['VECTOR_BUCKET_NAME']}")
print(f"👤 ユーザーID: {USER_ID}")

In [ ]:
# Bedrockモデルのセットアップ

session = boto3.Session(region_name=AWS_REGION)
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-3-5-sonnet-20241022-v2:0",
    boto_session=session
)

# 旅行アシスタント用のシステムプロンプト
TRAVEL_ASSISTANT_PROMPT = """あなたは永続的なメモリを持つ専門のAI旅行アシスタントです。

あなたの能力：
- **パーソナライズされた推奨**: ユーザーの好みに基づいて提案を調整
- **マルチモーダル分析**: 写真、ドキュメント、テキストを処理
- **セッション間メモリ**: 以前の会話からの好みとコンテキストを記憶
- **文化的専門知識**: 目的地、料理、現地体験についての洞察を提供

メモリ使用ガイドライン：
1. **常に開始時**にユーザーに関する関連するメモリを取得
2. **重要な情報を保存**：
   - 旅行の好み（食べ物、活動、宿泊スタイル）
   - 食事制限や要件
   - 予算の考慮事項
   - 興味のある目的地
   - 過去の旅行体験
3. 複数の対話にわたって**コンテキストを構築**
4. 関連する場合は**以前の会話を参照**

コンテンツを分析する際：
1. メモリからユーザーの旅行の好みを取得
2. 新しいコンテンツ（写真、ドキュメントなど）を分析
3. 重要な洞察と詳細を保存
4. 両方に基づいてパーソナライズされた推奨を提供

常に熱心で、親切で、文化的に敏感に接してください。
"""

# 旅行アシスタントエージェントの作成
travel_assistant = Agent(
    model=bedrock_model,
    tools=[
        s3_vector_memory,  # 永続メモリ
        image_reader,      # 写真分析
        video_reader_local,      # 動画分析
        file_read         # ドキュメント処理
    ],
    system_prompt=TRAVEL_ASSISTANT_PROMPT
)

print("✅ 旅行アシスタントエージェントを作成しました！")

## セッション1: 旅行の好みの確立

最初のセッションで、ユーザーは自分の旅行スタイルと好みを共有します。

In [ ]:
response = travel_assistant(
    f"""こんにちは！次の旅行を計画していて、旅行の好みを共有したいと思いました。
    
    私が好きなこと：
    - **建築**: 現代建築、特にアール・ヌーヴォーとモダニストスタイルに魅了されています
    - **食べ物**: ベジタリアン料理を好み、地元の食品市場を探索するのが好きです
    - **持続可能性**: 持続可能な旅行を心がけています - 公共交通機関、エコフレンドリーなホテル、地元企業の支援
    - **活動**: ビーチ/リゾート休暇よりも、ウォーキングツアー、写真、文化的体験を楽しみます
    - **ペース**: 各場所を本当に体験する時間がある、リラックスしたペースを好みます
    
    今後の会話のために、これらの好みを覚えておいてください。
    
    ユーザーID: {USER_ID}"""
)

print(response)

## セッション2: 目的地の写真を分析

数日後、ユーザーは夢の目的地の写真を共有します。

**新しいセッションをシミュレート** - エージェントには会話履歴がなく、ベクトルメモリのみがあります。

In [ ]:
image = "output/professional_travel_photography_of_alcatraz.png"
video = "output/san-francisco-tour.mp4"
document = "output/san-francisco-itinerary.txt"

In [ ]:
response = travel_assistant(
    f"""訪れたい場所の素晴らしい写真を見つけました！
    
    次の画像を分析してください: {image}
    
    ユーザーID: {USER_ID}"""
)

print(response)

## セッション3: 旅行旅程の処理

ユーザーは旅行を予約し、パーソナライズされた提案のために旅程を共有します。

**別の新しいセッション** - セッション間メモリを再度テストします。

In [ ]:
# 別のセッションのために会話履歴を再度クリア
travel_assistant.messages.clear()

print("🆕 別の新しいセッション - 会話履歴を再度クリア")
print("🧠 メモリはすべてのセッションにわたって永続します！")
print(f"💬 履歴内のメッセージ数: {len(travel_assistant.messages)}")

In [ ]:
response = travel_assistant(
    f"""素晴らしいニュースです！旅行を予約し、旅程の準備ができました。
    
    次の旅程を確認してください: {document}
    
    ユーザーID: {USER_ID}"""
)

print(response)

## セッション4: パーソナライズされた推奨

数日後、ユーザーはレストランの推奨を求めます。

**最後の新しいセッション** - エージェントはすべてを覚えているはずです。

In [ ]:
# 会話履歴をもう一度クリア
travel_assistant.messages.clear()

print("🆕 最後の新しいセッション - 会話履歴をクリア")
print("🧠 ベクトルメモリからすべてを覚えているか確認しましょう！")
print(f"💬 履歴内のメッセージ数: {len(travel_assistant.messages)}")

In [ ]:
response = travel_assistant(
    f"""数日後に旅行に出発します！
    
    試すべきレストランをいくつか推奨してもらえますか？
    
    ユーザーID: {USER_ID}"""
)

print(response)

## セッション5: 動画分析

ユーザーはオンラインで見つけた旅行動画を共有し、パーソナライズされた洞察を求めます。

**別の新しいセッション** - メモリコンテキストでの動画分析を実証します。

In [ ]:
# 動画分析セッションのために会話履歴をクリア
travel_assistant.messages.clear()

print("🆕 動画分析のための新しいセッション")
print("🧠 エージェントはメモリを使用してパーソナライズされた動画の洞察を提供します")
print(f"💬 履歴内のメッセージ数: {len(travel_assistant.messages)}")

In [ ]:
response = travel_assistant(
    f"""あなたの意見を聞きたい動画を見つけました！
    
    次の動画を分析してください: {video}
    
    ユーザーID: {USER_ID}"""
)

print(response)

## メモリの検査

エージェントがベクトルメモリに保存した内容を確認しましょう。

In [ ]:
# すべてのメモリを一覧表示
result = s3_vector_memory(
    action="list",
    user_id=USER_ID,
    top_k=20
)

print(f"📊 保存されたメモリの合計: {result['total_found']}")
print("\n" + "="*80)
print("保存されたメモリ:")
print("="*80)

for i, mem in enumerate(result.get('memories', []), 1):
    content = mem.get('memory', '')
    timestamp = mem.get('created_at', 'N/A')
    
    print(f"\n{i}. [{timestamp}]")
    print(f"   {content[:200]}..." if len(content) > 200 else f"   {content}")
    print("-" * 80)

## セマンティックメモリ検索

セマンティック検索機能をテストします。

In [ ]:
# 建築関連のメモリを検索
result = s3_vector_memory(
    action="retrieve",
    query="建築の好み",
    user_id=USER_ID,
    top_k=5
)

print("🔍 検索: '建築の好みとガウディの建物'")
print("\nトップ結果:")
for i, mem in enumerate(result.get('memories', []), 1):
    print(f"\n{i}. 類似度: {mem.get('similarity', 'N/A')}")
    print(f"   {mem.get('memory', '')}")

In [ ]:
# 食べ物関連のメモリを検索
result = s3_vector_memory(
    action="retrieve",
    query="食べ物の好みとレストラン",
    user_id=USER_ID,
    top_k=5
)

print("🔍 検索: 'ベジタリアン料理の好みとレストラン'")
print("\nトップ結果:")
for i, mem in enumerate(result.get('memories', []), 1):
    print(f"\n{i}. 類似度: {mem.get('similarity', 'N/A')}")
    print(f"   {mem.get('memory', '')}")

### メモリフロー

**メモリの保存:**
1. エージェントが会話から重要な情報を抽出
2. `s3_vector_memory(action="store", content=..., user_id=...)`を呼び出し
3. テキストがNova経由で埋め込みに変換
4. 埋め込み + メタデータがS3 Vectorsに保存

**メモリの取得:**
1. エージェントが応答のためのコンテキストを必要とする
2. `s3_vector_memory(action="retrieve", query=..., user_id=...)`を呼び出し
3. クエリが埋め込みに変換
4. S3 Vectorsで類似性検索
5. 最も関連性の高いTop-Kメモリが返される
6. エージェントがメモリを使用して応答を通知

### コスト最適化

- **埋め込み**: 1Kトークンあたり約$0.0001（Nova）
- **ストレージ**: S3 Vectorsの価格（コスト最適化）
- **クエリ**: 高速なサブ秒レベルの取得
- **ヒント**: コストと関連性のバランスを取るために適切な`top_k`値を使用

## クリーンアップ（オプション）

継続的な料金を避けるため、このデモ用に作成したS3 Vectorインデックスとバケットを削除できます。

In [ ]:
import boto3

# S3 Vectorsクライアントの初期化
s3vectors_client = boto3.client('s3vectors', region_name=AWS_REGION)

# ベクトルインデックスの削除
try:
    print(f"インデックスを削除中: {os.environ['VECTOR_INDEX_NAME']}...")
    s3vectors_client.delete_index(
        VectorBucketName=os.environ['VECTOR_BUCKET_NAME'],
        IndexName=os.environ['VECTOR_INDEX_NAME']
    )
    print(f"✅ インデックス '{os.environ['VECTOR_INDEX_NAME']}' を正常に削除しました")
except Exception as e:
    print(f"❌ インデックス削除エラー: {e}")

In [ ]:
# ベクトルバケットの削除
try:
    print(f"バケットを削除中: {os.environ['VECTOR_BUCKET_NAME']}...")
    s3vectors_client.delete_vector_bucket(
        VectorBucketName=os.environ['VECTOR_BUCKET_NAME']
    )
    print(f"✅ バケット '{os.environ['VECTOR_BUCKET_NAME']}' を正常に削除しました")
except Exception as e:
    print(f"❌ バケット削除エラー: {e}")

print("\n⚠️  注意: 削除が完了するまで数分かかる場合があります。")

## まとめ

このデモでは、以下を学習しました：

✅ 永続的なメモリを持つパーソナライズされたAIアシスタントの構築方法

✅ 会話履歴なしでのセッション間コンテキスト保持

✅ マルチモーダルコンテンツ分析（テキスト、画像、PDF）

✅ セマンティックメモリ検索と取得

✅ 複数の対話にわたる段階的なパーソナライゼーション

✅ AWSサービスを使用した本番環境対応のアーキテクチャ

### リソース

- [Amazon S3 Vectorsドキュメント](https://docs.aws.amazon.com/AmazonS3/latest/userguide/s3-vectors.html)
- [Amazon Nova Embeddings](https://aws.amazon.com/blogs/aws/amazon-nova-multimodal-embeddings-now-available-in-amazon-bedrock/)
- [Strands Agents SDK](https://github.com/awslabs/strands)

素晴らしい構築を！ 🚀